# 🚀 Deriv Hybrid Predictor - Colab API (FIXED v2)

**Run the entire prediction engine on Google Colab's free GPU**

---

## 📋 Before You Start

1. **Enable GPU**: Runtime → Change runtime type → GPU
2. **Get ngrok token**: https://dashboard.ngrok.com/get-started/your-authtoken
3. **Upload project to Drive**: Upload the `der` folder to Google Drive

---

## ▶️ Run All Cells Below

## 1️⃣ Mount Google Drive

In [ ]:
from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')

print("✅ Google Drive mounted")

## 2️⃣ Locate Project

In [ ]:
import os
import sys

# Try common locations
possible_paths = [
    '/content/drive/MyDrive/der',
    '/content/drive/MyDrive/Deriv_Predictor',
    '/content/drive/MyDrive/2026/der',
    '/content/drive/My Drive/der',
    '/content/drive/My Drive/Deriv_Predictor'
]

PROJECT_PATH = None

# Find project
for path in possible_paths:
    if os.path.exists(path):
        PROJECT_PATH = path
        break

if PROJECT_PATH:
    print(f"✅ Project found at: {PROJECT_PATH}")
    os.chdir(PROJECT_PATH)
    print(f"📁 Working directory: {os.getcwd()}")
else:
    print("❌ Project not found!")
    print("\n📍 Please upload the 'der' folder to one of these locations:")
    for path in possible_paths:
        print(f"   - {path}")
    
    # Let user set custom path
    custom_path = input("\nEnter custom path (or press Enter to skip): ").strip()
    if custom_path and os.path.exists(custom_path):
        PROJECT_PATH = custom_path
        os.chdir(PROJECT_PATH)
        print(f"✅ Using custom path: {PROJECT_PATH}")
    else:
        raise Exception("Project path not found. Please upload the project to Google Drive first.")

## 3️⃣ Install Dependencies

In [ ]:
print("📦 Installing dependencies...")

# Install dependencies
!pip install -q tensorflow scikit-learn pandas numpy matplotlib seaborn
!pip install -q fastapi uvicorn pyngrok nest-asyncio

print("✅ Dependencies installed")

## 4️⃣ Check GPU Availability

In [ ]:
import tensorflow as tf

# Check GPU
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print(f"✅ GPU available: {gpus[0].name}")
else:
    print("⚠️ No GPU found!")
    print("   Go to: Runtime → Change runtime type → GPU")

## 5️⃣ Load Core Engine

In [ ]:
import sys
import os

# Add project to Python path
if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

# Import core engine
try:
    from core import HybridEngine
    from core.config import DERIV_CONFIG, BUFFER_SIZE, RETRAIN_CONFIG
    
    print("✅ Core engine imported successfully")
    
except ImportError as e:
    print(f"❌ Failed to import core engine: {e}")
    raise

## 6️⃣ Initialize Engine

In [ ]:
from datetime import datetime

# Create model save directory
MODEL_DIR = os.path.join(PROJECT_PATH, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

print("🧠 Initializing Hybrid Engine...")

# Initialize engine
engine = HybridEngine(verbose=True)

try:
    lstm_path = os.path.join(MODEL_DIR, 'lstm_model.h5')
    if os.path.exists(lstm_path):
        engine.lstm_engine.load_model(lstm_path)
        print("📦 Loaded existing LSTM model")
        engine.is_trained = True
except Exception as e:
    print(f"⚠️ No existing models found (will train from scratch)")

## 7️⃣ Create FastAPI Server

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Dict, Optional
import nest_asyncio
import traceback

# Allow nested event loops (required for Colab)
nest_asyncio.apply()

# Create FastAPI app
app = FastAPI(
    title="Deriv Hybrid Predictor API",
    version="1.0.0"
)

# Request/Response models
class Tick(BaseModel):
    timestamp: int
    quote: float
    symbol: str

class PredictRequest(BaseModel):
    ticks: List[Tick]

class PredictResponse(BaseModel):
    prediction: Optional[Dict]
    status: Dict
    timestamp: str

# Endpoints
@app.get("/")
async def root():
    status = engine.get_status()
    return {
        "name": "Deriv Hybrid Predictor",
        "status": "running",
        "gpu_available": len(tf.config.list_physical_devices('GPU')) > 0,
        "engine_trained": status.get('engine', {}).get('is_trained', False),
        "tick_count": status.get('engine', {}).get('tick_count', 0)
    }

@app.post("/predict", response_model=PredictResponse)
async def predict(request: PredictRequest):
    """
    Get prediction for given ticks
    """
    try:
        # Add ticks to engine
        for tick in request.ticks:
            # Handle Pydantic v1 vs v2 compatibility
            if hasattr(tick, 'model_dump'):
                tick_data = tick.model_dump()
            elif hasattr(tick, 'dict'):
                tick_data = tick.dict()
            else:
                tick_data = vars(tick)
            
            engine.add_tick(tick_data)
        
        # Train if needed
        if not engine.is_trained and engine.tick_count >= 500:
            print(f"🚀 Starting initial training (Ticks: {engine.tick_count})...")
            try:
                # WRAPPED IN TRY/EXCEPT SO SERVER DOES NOT CRASH
                results = engine.initial_train()
                print(f"✅ Training complete: {results['lstm_accuracy']:.2%}")
                
                # Save models
                engine.lstm_engine.save_model(os.path.join(MODEL_DIR, 'lstm_model.h5'))
            except Exception as train_error:
                print(f"❌ Training failed: {train_error}")
                traceback.print_exc()
                # We swallow the error to keep the server running
                # The status will show untrained
        
        # Get prediction
        prediction = None
        if engine.is_trained:
            prediction = engine.predict_next_tick()
        
        # Get proper status format
        full_status = engine.get_status()
        api_status = {
            "is_trained": full_status['engine']['is_trained'],
            "tick_count": full_status['engine']['tick_count'],
            "gpu_available": len(tf.config.list_physical_devices('GPU')) > 0
        }
        
        return PredictResponse(
            prediction=prediction,
            status=api_status,
            timestamp=datetime.now().isoformat()
        )
    
    except Exception as e:
        print(f"❌ Predict Error: {e}")
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/status")
async def get_status():
    return engine.get_status()

@app.post("/retrain")
async def force_retrain():
    if not engine.is_trained:
        raise HTTPException(400, "Engine not trained yet")
    
    print("🔄 Forcing retrain...")
    results = engine.retrain()
    
    # Save models
    engine.lstm_engine.save_model(os.path.join(MODEL_DIR, 'lstm_model.h5'))
    
    return {
        "message": "Retrain complete",
        "results": results
    }

print("✅ FastAPI server created")

## 8️⃣ Setup ngrok (Public URL)

In [ ]:
from pyngrok import ngrok
import getpass

# Get ngrok token
print("🔑 Get your ngrok token from: https://dashboard.ngrok.com/get-started/your-authtoken")
ngrok_token = getpass.getpass("Enter your ngrok authtoken: ")

# Set token
ngrok.set_auth_token(ngrok_token)

print("✅ ngrok token set")

## 9️⃣ Start Server (FIXED)

In [ ]:
import uvicorn
from threading import Thread
import time

# Configure uvicorn manually to avoid loop_factory issue with nest_asyncio
config = uvicorn.Config(
    app=app, 
    host="0.0.0.0", 
    port=8000, 
    log_level="info",
    loop="asyncio"
)
server = uvicorn.Server(config)

# Run server within asyncio loop manually in a thread
def run_uvicorn():
    import asyncio
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(server.serve())

server_thread = Thread(target=run_uvicorn, daemon=True)
server_thread.start()

# Wait for server to start
time.sleep(3)

# Create ngrok tunnel
try:
    tunnel = ngrok.connect(8000)
    public_url = tunnel.public_url
except Exception as e:
    # Fallback/retry
    print(f"ngrok connect error: {e}, retrying...")
    time.sleep(2)
    tunnel = ngrok.connect(8000)
    public_url = tunnel.public_url

print("=" * 70)
print("🎉 SERVER RUNNING!")
print("=" * 70)
print(f"\n📡 Public URL: {public_url}")
print(f"\n🔗 COPY THIS URL and use it in your local backend!")
print(f"\n📝 In backend/main.py, set:")
print(f"   COLAB_URL = \"{public_url}\"")
print(f"\n📚 API Docs: {public_url}/docs")
print(f"\n✅ Test endpoint: {public_url}/")
print("\n" + "=" * 70)

## 🔟 Monitor Server (FIXED)

In [ ]:
import time
from IPython.display import clear_output

# Monitor loop
try:
    while True:
        clear_output(wait=True)
        
        # Get full status dict
        status = engine.get_status()
        engine_stats = status.get('engine', {})
        ensemble_stats = status.get('ensemble', {})
        
        print("=" * 70)
        print("📊 DERIV PREDICTOR - LIVE STATUS")
        print("=" * 70)
        print(f"\n🔗 Public URL: {public_url}")
        print(f"\n🧠 Engine Status:")
        print(f"   Trained: {engine_stats.get('is_trained', False)}")
        print(f"   Tick count: {engine_stats.get('tick_count', 0)}")
        print(f"   Predictions: {engine_stats.get('prediction_count', 0)}")
        
        if engine_stats.get('is_trained', False):
            print(f"\n📈 Performance:")
            print(f"   Accuracy (50): {ensemble_stats.get('accuracy_50', 0):.2%}")
            print(f"   Accuracy (200): {ensemble_stats.get('accuracy_200', 0):.2%}")
        
        print(f"\n⏰ Last update: {datetime.now().strftime('%H:%M:%S')}")
        print("\n💡 Press Ctrl+C to stop monitoring (server keeps running)")
        print("=" * 70)
        
        time.sleep(5)

except KeyboardInterrupt:
    print("\n✅ Monitoring stopped (server still running)")